In [ ]:
# Football Player Database — SQL Analysis

**A 17-table normalized SQLite database of 27,343 football players, analyzed SQL-first.**

## Dataset
- 27,343 players, 108,639 player-seasons (~4 per player)
- Star schema: hubs `players` and `player_seasons`; 5 vertically-partitioned stat tables;
  dimensions `clubs → divisions`, `nations`, `positions`, `seasons`, `agents`

## What I built (in order)
1. **Schema map** — traced foreign keys from SQLite's catalog; documented the star schema and join paths
2. **Player-season overview** — JOIN from the fact table outward; 1:1 stats joins (no fan-out)
3. **Top scorers per season** — `RANK() OVER (PARTITION BY season ORDER BY goals_per_90)`, min-900-min filter
4. **Injury downtime** — `julianday()` date math; NULL handling for ongoing injuries
5. **Transfer spend by division** — multi-hop JOIN; self-referential FK (`to_club_id`)
6. **Cumulative injuries** — `SUM(COUNT()) OVER (ORDER BY ...)`; caught a reversed-ordering bug
7. **Capstone: wage vs. performance** — see below

## Capstone: does wage buy performance?

**Naive answer:** barely. Goals/90 is nearly flat across wage bands (~0.01–0.03 gap).

**Diagnosis:** goals are position-biased — ST 0.349 vs GK 0.000. Pooling across
positions dilutes any wage signal.

**Corrected outcome:** `ability_score` (position-adjusted) rises **+74 points per wage band**
(520 → 609 → 683). Wage *does* buy quality.

**Proof of confound:** assists rank M > AM > DM > ST (the *opposite* of goals) —
so no single stat measures "performance."

**Goalkeepers:** high-wage GKs make more saves (3.29 vs 2.66) and save a higher ratio
(75.7% vs 70.1%) — but prevent *fewer* expected goals (−0.006 vs +0.075). They face
harder shots; only the quality-adjusted metric tells the truth.

## Thesis
**Wage buys environment, not edge.** Higher wage tracks higher overall ability and more
raw volume (goals, saves), but does **not** improve quality-adjusted overperformance
(xG, xGP). Expensive players are in better systems, not better relative to expectation.
Any wage–performance analysis using raw volume metrics will overstate expensive players.

## Debugging notes
- **Caught a silent ordering bug:** the cumulative-injury window initially counted
  *backward* in time — `season_order` ran descending. Verified against known rows and
  added `DESC`. Silent ordering bugs in window functions produce plausible but inverted output.
- **Verified column locations before joining:** stats are split across 5 themed tables;
  goals live in `player_stats_attacking`, rating in `player_stats_general`.

## Limitations
- Event tables (transfers, injuries) are **100% populated** — synthetic by design.
  Reported *patterns* (rank, duration, distribution), not *frequencies*.
- No birthdate column; age taken from `player_seasons.age`.
- `player_positions` is player-level, not season-level → position breakdown counts players.
- Differences across wage bands are small (0.01–0.03 goals/90); results are directional,
  not significance-tested.

In [1]:
!pip install ipython-sql prettytable -q

import prettytable
prettytable.DEFAULT = 'DEFAULT'
%load_ext sql

In [2]:
%sql sqlite:////kaggle/input/datasets/tomersolo/sql-practice-football-players/sqlpractice.sqlite

In [3]:
%%sql
SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;

 * sqlite:////kaggle/input/datasets/tomersolo/sql-practice-football-players/sqlpractice.sqlite
Done.


name
agents
clubs
contracts
divisions
injuries
nations
player_positions
player_seasons
player_stats_attacking
player_stats_defending


In [4]:
%%sql
SELECT m.name AS child_table,
       f."from" AS child_col,
       f."table" AS parent_table,
       f."to" AS parent_col
FROM sqlite_master m
JOIN pragma_foreign_key_list(m.name) f
WHERE m.type='table'
ORDER BY child_table, parent_table;

 * sqlite:////kaggle/input/datasets/tomersolo/sql-practice-football-players/sqlpractice.sqlite
Done.


child_table,child_col,parent_table,parent_col
clubs,division_id,divisions,division_id
contracts,club_id,clubs,club_id
contracts,player_id,players,player_id
injuries,player_id,players,player_id
injuries,season_id,seasons,season_id
player_positions,player_id,players,player_id
player_positions,position_id,positions,position_id
player_seasons,club_id,clubs,club_id
player_seasons,player_id,players,player_id
player_seasons,season_id,seasons,season_id


In [8]:
%%sql
SELECT m.name AS tbl, p.name AS col
FROM sqlite_master m
JOIN pragma_table_info(m.name) p
WHERE m.type='table'
  AND m.name IN ('players','clubs','seasons','player_seasons','player_stats_general')
ORDER BY m.name, p.cid;

 * sqlite:////kaggle/input/datasets/tomersolo/sql-practice-football-players/sqlpractice.sqlite
Done.


tbl,col
clubs,club_id
clubs,club_name
clubs,division_id
player_seasons,player_season_id
player_seasons,player_id
player_seasons,season_id
player_seasons,club_id
player_seasons,age
player_seasons,mins
player_seasons,starts


In [9]:
%%sql
SELECT m.name AS tbl, p.name AS col
FROM sqlite_master m
JOIN pragma_table_info(m.name) p
WHERE m.type='table' AND m.name IN ('player_stats_attacking','player_stats_defending')
ORDER BY m.name, p.cid;

 * sqlite:////kaggle/input/datasets/tomersolo/sql-practice-football-players/sqlpractice.sqlite
Done.


tbl,col
player_stats_attacking,player_season_id
player_stats_attacking,goals
player_stats_attacking,goals_per_90
player_stats_attacking,xg
player_stats_attacking,xg_per_90
player_stats_attacking,non_penalty_xg
player_stats_attacking,non_penalty_xg_per_90
player_stats_attacking,xg_overperformance
player_stats_attacking,shots
player_stats_attacking,shots_per_90


In [10]:
%%sql
SELECT p.name,
       c.club_name,
       s.season_label,
       ps.starts + ps.sub_appearances AS apps,
       ps.mins,
       psa.goals,
       psa.assists,
       psg.average_rating
FROM player_seasons ps
JOIN players p                ON ps.player_id = p.player_id
JOIN clubs c                  ON ps.club_id   = c.club_id
JOIN seasons s                ON ps.season_id = s.season_id
JOIN player_stats_general psg ON psg.player_season_id = ps.player_season_id
JOIN player_stats_attacking psa ON psa.player_season_id = ps.player_season_id
ORDER BY s.season_label, psa.goals DESC
LIMIT 20;

 * sqlite:////kaggle/input/datasets/tomersolo/sql-practice-football-players/sqlpractice.sqlite
Done.


name,club_name,season_label,apps,mins,goals,assists,average_rating
Dušan Vlahović,Juventus,2022/23,42,3532,34,6,7.53
Nesto Groen,Tottori,2022/23,44,3818,33,6,7.54
Jannis Hampl,ASV Neumarkt,2022/23,52,3492,33,3,7.58
Norman Bassette,Santos Laguna,2022/23,43,3588,31,6,6.85
Axel Eboki-Poh,Argentino (Rosario),2022/23,45,3906,31,11,7.39
Vitor Roque,Cork City,2022/23,45,4033,31,8,7.68
Amir Shahim,Persepolis,2022/23,61,4912,30,3,7.98
Gerhard Callebaut,RSC Anderlecht,2022/23,41,3533,29,5,6.61
Andy Müller,Real Cartagena,2022/23,44,3531,29,12,7.36
Fabian Shahaj,Sainte-Geneviève Sports,2022/23,47,4055,29,13,7.4


In [11]:
%%sql
SELECT season_label, name, club_name, goals,
       RANK() OVER (PARTITION BY season_label ORDER BY goals DESC) AS goal_rank
FROM (
    SELECT s.season_label, p.name, c.club_name, psa.goals
    FROM player_seasons ps
    JOIN players p                  ON ps.player_id = p.player_id
    JOIN clubs c                    ON ps.club_id   = c.club_id
    JOIN seasons s                  ON ps.season_id = s.season_id
    JOIN player_stats_attacking psa ON psa.player_season_id = ps.player_season_id
) t
WHERE goals > 0
ORDER BY season_label, goal_rank
LIMIT 30;

 * sqlite:////kaggle/input/datasets/tomersolo/sql-practice-football-players/sqlpractice.sqlite
Done.


season_label,name,club_name,goals,goal_rank
2022/23,Dušan Vlahović,Juventus,34,1
2022/23,Nesto Groen,Tottori,33,2
2022/23,Jannis Hampl,ASV Neumarkt,33,2
2022/23,Norman Bassette,Santos Laguna,31,4
2022/23,Axel Eboki-Poh,Argentino (Rosario),31,4
2022/23,Vitor Roque,Cork City,31,4
2022/23,Amir Shahim,Persepolis,30,7
2022/23,Gerhard Callebaut,RSC Anderlecht,29,8
2022/23,Andy Müller,Real Cartagena,29,8
2022/23,Fabian Shahaj,Sainte-Geneviève Sports,29,8


In [12]:
%%sql
SELECT m.name AS tbl, p.name AS col
FROM sqlite_master m
JOIN pragma_table_info(m.name) p
WHERE m.type='table' AND m.name IN ('players','transfers','injuries','contracts')
ORDER BY m.name, p.cid;

 * sqlite:////kaggle/input/datasets/tomersolo/sql-practice-football-players/sqlpractice.sqlite
Done.


tbl,col
contracts,contract_id
contracts,player_id
contracts,club_id
contracts,start_date
contracts,end_date
contracts,weekly_wage_eur
contracts,contract_value_eur
injuries,injury_id
injuries,player_id
injuries,season_id


In [13]:
%%sql
SELECT p.name,
       COUNT(*) AS injury_count,
       SUM(julianday(i.end_date) - julianday(i.start_date)) AS total_days_out,
       ROUND(AVG(julianday(i.end_date) - julianday(i.start_date)), 1) AS avg_days_per_injury
FROM injuries i
JOIN players p ON p.player_id = i.player_id
WHERE i.end_date IS NOT NULL AND i.start_date IS NOT NULL
GROUP BY p.player_id, p.name
ORDER BY total_days_out DESC
LIMIT 20;

 * sqlite:////kaggle/input/datasets/tomersolo/sql-practice-football-players/sqlpractice.sqlite
Done.


name,injury_count,total_days_out,avg_days_per_injury
Jerson Bernal,3,770.0,256.7
Nacer Belaïd,3,727.0,242.3
Tomás Tavares,3,683.0,227.7
Adam ten Voorde,3,632.0,210.7
Emanuele Magnino,3,556.0,185.3
Mohammad Hossein Asgharzadeh,3,541.0,180.3
Kilian Sauck,2,524.0,262.0
Mondli Manana,3,522.0,174.0
Femi Seriki,4,518.0,129.5
Kamel Hmeisheh,3,517.0,172.3


In [14]:
%%sql
SELECT d.division_name,
       COUNT(*) AS transfers_in,
       ROUND(SUM(t.transfer_fee_eur), 0) AS total_fees_paid,
       ROUND(AVG(t.transfer_fee_eur), 0) AS avg_fee,
       ROUND(MAX(t.transfer_fee_eur), 0) AS biggest_fee
FROM transfers t
JOIN clubs c      ON t.to_club_id = c.club_id
JOIN divisions d  ON c.division_id = d.division_id
WHERE t.transfer_fee_eur IS NOT NULL AND t.transfer_fee_eur > 0
GROUP BY d.division_name
ORDER BY total_fees_paid DESC
LIMIT 15;

 * sqlite:////kaggle/input/datasets/tomersolo/sql-practice-football-players/sqlpractice.sqlite
Done.


division_name,transfers_in,total_fees_paid,avg_fee,biggest_fee
English Premier Division,127,284769000.0,2242276.0,5226000.0
Italian Serie A,138,200811000.0,1455152.0,4597000.0
Spanish First Division,128,185625000.0,1450195.0,4051000.0
Sky Bet Championship,163,168200000.0,1031902.0,4108000.0
Bundesliga,129,162902000.0,1262806.0,4567000.0
Ligue 1 Uber Eats,111,162220000.0,1461441.0,4657000.0
Argentine Premier Division,150,137445000.0,916300.0,3637000.0
Brazilian National First Division,113,127736000.0,1130407.0,3982000.0
Spanish Second Division,168,117781000.0,701077.0,3292000.0
Portuguese Premier League,116,110748000.0,954724.0,3459000.0


In [15]:
%%sql
SELECT
  (SELECT COUNT(*) FROM players)        AS all_players,
  (SELECT COUNT(DISTINCT player_id) FROM player_seasons) AS players_with_seasons,
  (SELECT COUNT(DISTINCT player_id) FROM transfers)      AS players_transferred,
  (SELECT COUNT(*) FROM player_seasons) AS player_season_rows;

 * sqlite:////kaggle/input/datasets/tomersolo/sql-practice-football-players/sqlpractice.sqlite
Done.


all_players,players_with_seasons,players_transferred,player_season_rows
27343,27343,27343,108639


In [17]:
%%sql
SELECT p.name,
       s.season_label,
       COUNT(i.injury_id) AS injuries_this_season,
       SUM(COUNT(i.injury_id)) OVER (
           PARTITION BY i.player_id
           ORDER BY s.season_order DESC
       ) AS cumulative_injuries
FROM injuries i
JOIN players p ON p.player_id = i.player_id
JOIN seasons s ON s.season_id = i.season_id
GROUP BY i.player_id, p.name, s.season_label, s.season_order
ORDER BY p.name, s.season_order DESC
LIMIT 30;

 * sqlite:////kaggle/input/datasets/tomersolo/sql-practice-football-players/sqlpractice.sqlite
Done.


name,season_label,injuries_this_season,cumulative_injuries
Aaron Abass,2023/24,1,1
Aaron Abass,2024/25,1,2
Aaron Abass,2025/26,1,3
Aaron Bott,2022/23,1,1
Aaron Chileshe,2022/23,1,1
Aaron Chileshe,2023/24,1,2
Aaron Ciammaglichella,2025/26,1,1
Aaron Clarke,2022/23,1,1
Aaron Clarke,2025/26,1,2
Aaron Connolly,2022/23,1,1


In [19]:
%%sql
SELECT d.division_name,
       COUNT(DISTINCT ct.player_id) AS players,
       ROUND(AVG(ct.weekly_wage_eur),0) AS avg_wage,
       ROUND(MIN(ct.weekly_wage_eur),0) AS min_wage,
       ROUND(MAX(ct.weekly_wage_eur),0) AS max_wage,
       COUNT(DISTINCT ct.weekly_wage_eur) AS distinct_wages
FROM contracts ct
JOIN clubs c     ON ct.club_id = c.club_id
JOIN divisions d ON c.division_id = d.division_id
WHERE ct.weekly_wage_eur IS NOT NULL
GROUP BY d.division_name
ORDER BY avg_wage DESC
LIMIT 15;

 * sqlite:////kaggle/input/datasets/tomersolo/sql-practice-football-players/sqlpractice.sqlite
Done.


division_name,players,avg_wage,min_wage,max_wage,distinct_wages
English Premier Division,428,121471.0,4314.0,216112.0,438
Italian Serie A,446,88412.0,4849.0,199054.0,449
Ligue 1 Uber Eats,418,87466.0,4475.0,197432.0,421
Spanish First Division,495,86499.0,5752.0,193103.0,501
Bundesliga,423,84693.0,5661.0,197798.0,428
Brazilian National First Division,446,74920.0,6876.0,188475.0,455
Jupiler Pro League,84,74281.0,3226.0,179934.0,84
Sky Bet Championship,637,65982.0,2899.0,184820.0,648
Portuguese Premier League,472,64650.0,2736.0,179702.0,479
Ecuadorian Serie A,24,63143.0,10234.0,145995.0,24


In [23]:
%%sql
WITH elite AS (
  SELECT 'English Premier Division' AS div UNION ALL
  SELECT 'Italian Serie A' UNION ALL
  SELECT 'Ligue 1 Uber Eats' UNION ALL
  SELECT 'Spanish First Division' UNION ALL
  SELECT 'Bundesliga'
),
player_wage AS (
  SELECT ct.player_id,
         AVG(ct.weekly_wage_eur) AS avg_wage
  FROM contracts ct
  GROUP BY ct.player_id
),
season_output AS (
  SELECT ps.player_id, ps.season_id, ps.mins,
         psa.goals_per_90, psa.xg_overperformance
  FROM player_seasons ps
  JOIN player_stats_attacking psa ON psa.player_season_id = ps.player_season_id
  WHERE ps.mins >= 900
)
SELECT s.season_label,
       CASE WHEN pw.avg_wage >= 100000 THEN 'high wage (≥100k)'
            WHEN pw.avg_wage >=  50000 THEN 'mid wage (50–100k)'
            ELSE 'low wage (<50k)' END AS wage_band,
       COUNT(*) AS player_seasons,
       ROUND(AVG(so.goals_per_90), 3)          AS avg_goals_per90,
       ROUND(AVG(so.xg_overperformance), 3)    AS avg_xg_overperf
FROM season_output so
JOIN player_wage pw ON pw.player_id = so.player_id
JOIN player_seasons ps ON ps.player_id = so.player_id AND ps.season_id = so.season_id
JOIN clubs c     ON ps.club_id = c.club_id
JOIN divisions d ON c.division_id = d.division_id
JOIN seasons s   ON s.season_id = so.season_id
WHERE d.division_name IN (SELECT div FROM elite)
GROUP BY s.season_label, wage_band
ORDER BY s.season_label, wage_band;

 * sqlite:////kaggle/input/datasets/tomersolo/sql-practice-football-players/sqlpractice.sqlite
Done.


season_label,wage_band,player_seasons,avg_goals_per90,avg_xg_overperf
2022/23,high wage (≥100k),453,0.149,0.069
2022/23,low wage (<50k),184,0.116,-0.014
2022/23,mid wage (50–100k),314,0.122,-0.041
2023/24,high wage (≥100k),496,0.141,0.025
2023/24,low wage (<50k),188,0.135,0.114
2023/24,mid wage (50–100k),347,0.133,0.003
2024/25,high wage (≥100k),557,0.143,0.071
2024/25,low wage (<50k),200,0.137,0.047
2024/25,mid wage (50–100k),397,0.133,-0.02
2025/26,high wage (≥100k),613,0.138,0.067


In [22]:
%%sql
WITH elite AS (
  SELECT 'English Premier Division' AS div UNION ALL
  SELECT 'Italian Serie A' UNION ALL
  SELECT 'Ligue 1 Uber Eats' UNION ALL
  SELECT 'Spanish First Division' UNION ALL
  SELECT 'Bundesliga'
),
pw AS (SELECT player_id, AVG(weekly_wage_eur) w FROM contracts GROUP BY player_id)
SELECT CASE WHEN pw.w>=100000 THEN 'high' WHEN pw.w>=50000 THEN 'mid' ELSE 'low' END AS wage_band,
       COUNT(*) AS n,
       ROUND(AVG(ps.ability_score),2) AS avg_ability,
       ROUND(MIN(ps.ability_score),1) AS min_ability,
       ROUND(MAX(ps.ability_score),1) AS max_ability
FROM player_seasons ps
JOIN pw ON pw.player_id = ps.player_id
JOIN clubs c     ON ps.club_id = c.club_id
JOIN divisions d ON c.division_id = d.division_id
WHERE d.division_name IN (SELECT div FROM elite)
GROUP BY wage_band
ORDER BY avg_ability DESC;

 * sqlite:////kaggle/input/datasets/tomersolo/sql-practice-football-players/sqlpractice.sqlite
Done.


wage_band,n,avg_ability,min_ability,max_ability
high,2636,683.06,307.9,768.3
mid,2282,609.08,260.6,768.0
low,1303,519.93,222.8,768.0


In [24]:
%%sql
SELECT * FROM positions LIMIT 3;

 * sqlite:////kaggle/input/datasets/tomersolo/sql-practice-football-players/sqlpractice.sqlite
Done.


position_id,role_code,side_code
1,AM,C
2,AM,L
3,AM,R


In [26]:
%%sql
SELECT pos.role_code,
       COUNT(*) AS n,
       ROUND(AVG(psa.goals_per_90), 3) AS avg_goals_per90
FROM player_seasons ps
JOIN player_positions pp ON pp.player_id = ps.player_id
JOIN positions pos       ON pos.position_id = pp.position_id
JOIN player_stats_attacking psa ON psa.player_season_id = ps.player_season_id
WHERE ps.mins >= 900
GROUP BY pos.role_code
ORDER BY avg_goals_per90 DESC;

 * sqlite:////kaggle/input/datasets/tomersolo/sql-practice-football-players/sqlpractice.sqlite
Done.


role_code,n,avg_goals_per90
ST,15045,0.349
AM,40784,0.214
M,34469,0.119
DM,15107,0.061
WB,9341,0.05
D,38522,0.038
GK,5441,0.0


In [27]:
%%sql
SELECT pos.role_code, COUNT(*) n,
       ROUND(AVG(psa.assists_per_90),3) avg_assists_per90
FROM player_seasons ps
JOIN player_positions pp ON pp.player_id = ps.player_id
JOIN positions pos       ON pos.position_id = pp.position_id
JOIN player_stats_attacking psa ON psa.player_season_id = ps.player_season_id
WHERE ps.mins >= 900
GROUP BY pos.role_code
ORDER BY avg_assists_per90 DESC;

 * sqlite:////kaggle/input/datasets/tomersolo/sql-practice-football-players/sqlpractice.sqlite
Done.


role_code,n,avg_assists_per90
M,34469,0.153
AM,40784,0.14
DM,15107,0.138
ST,15045,0.103
WB,9341,0.088
D,38522,0.065
GK,5441,0.0


In [28]:
%%sql
SELECT * FROM player_stats_goalkeeping LIMIT 1;

 * sqlite:////kaggle/input/datasets/tomersolo/sql-practice-football-players/sqlpractice.sqlite
Done.


player_season_id,saves_tipped,saves_parried,saves_held,saves_per_90,save_ratio,clean_sheets,clean_sheets_per_90,conceded_per_90,team_conceded,team_conceded_per_90,team_goals,team_goals_per_90,goals_conceded,penalties_saved,penalties_saved_ratio,expected_saves_ratio,expected_goals_prevented,expected_goals_prevented_per_90
1,0,0,0,0,0,0,0,0,0,1.45,33,1.59,0,0,0,0,0,0


In [29]:
%%sql
WITH elite AS (
  SELECT 'English Premier Division' AS div UNION ALL
  SELECT 'Italian Serie A' UNION ALL
  SELECT 'Ligue 1 Uber Eats' UNION ALL
  SELECT 'Spanish First Division' UNION ALL
  SELECT 'Bundesliga'
),
pw AS (SELECT player_id, AVG(weekly_wage_eur) w FROM contracts GROUP BY player_id)
SELECT CASE WHEN pw.w>=100000 THEN 'high' WHEN pw.w>=50000 THEN 'mid' ELSE 'low' END AS wage_band,
       COUNT(*) AS n,
       ROUND(AVG(psgk.saves_per_90), 2)                    AS avg_saves90,
       ROUND(AVG(psgk.clean_sheets_per_90), 3)             AS avg_cs90,
       ROUND(AVG(psgk.expected_goals_prevented_per_90), 3) AS avg_xgp90
FROM player_seasons ps
JOIN player_stats_goalkeeping psgk ON psgk.player_season_id = ps.player_season_id
JOIN player_positions pp ON pp.player_id = ps.player_id
JOIN positions pos       ON pos.position_id = pp.position_id
JOIN pw ON pw.player_id = ps.player_id
JOIN clubs c     ON ps.club_id = c.club_id
JOIN divisions d ON c.division_id = d.division_id
WHERE pos.role_code = 'GK'
  AND d.division_name IN (SELECT div FROM elite)
  AND ps.mins >= 900
GROUP BY wage_band
ORDER BY avg_xgp90 DESC;

 * sqlite:////kaggle/input/datasets/tomersolo/sql-practice-football-players/sqlpractice.sqlite
Done.


wage_band,n,avg_saves90,avg_cs90,avg_xgp90
low,70,2.66,0.248,0.075
mid,120,2.78,0.31,0.054
high,147,3.29,0.294,-0.006


In [30]:
%%sql
WITH elite AS (
  SELECT 'English Premier Division' AS div UNION ALL
  SELECT 'Italian Serie A' UNION ALL
  SELECT 'Ligue 1 Uber Eats' UNION ALL
  SELECT 'Spanish First Division' UNION ALL
  SELECT 'Bundesliga'
),
pw AS (SELECT player_id, AVG(weekly_wage_eur) w FROM contracts GROUP BY player_id)
SELECT CASE WHEN pw.w>=100000 THEN 'high' WHEN pw.w>=50000 THEN 'mid' ELSE 'low' END AS wage_band,
       COUNT(*) AS n,
       ROUND(AVG(psgk.save_ratio), 3)                      AS avg_save_ratio,
       ROUND(AVG(psgk.expected_goals_prevented_per_90), 3) AS avg_xgp90,
       ROUND(AVG(psgk.conceded_per_90), 2)                 AS avg_conceded90
FROM player_seasons ps
JOIN player_stats_goalkeeping psgk ON psgk.player_season_id = ps.player_season_id
JOIN player_positions pp ON pp.player_id = ps.player_id
JOIN positions pos       ON pos.position_id = pp.position_id
JOIN pw ON pw.player_id = ps.player_id
JOIN clubs c     ON ps.club_id = c.club_id
JOIN divisions d ON c.division_id = d.division_id
WHERE pos.role_code = 'GK' AND d.division_name IN (SELECT div FROM elite) AND ps.mins >= 900
GROUP BY wage_band
ORDER BY avg_save_ratio DESC;

 * sqlite:////kaggle/input/datasets/tomersolo/sql-practice-football-players/sqlpractice.sqlite
Done.


wage_band,n,avg_save_ratio,avg_xgp90,avg_conceded90
high,147,0.757,-0.006,1.52
mid,120,0.739,0.054,1.22
low,70,0.701,0.075,1.17


In [31]:
%%sql
SELECT pos.role_code,
       COUNT(*) AS n,
       ROUND(AVG(psa.assists_per_90), 3) AS avg_assists_per90
FROM player_seasons ps
JOIN player_positions pp        ON pp.player_id = ps.player_id
JOIN positions pos              ON pos.position_id = pp.position_id
JOIN player_stats_attacking psa ON psa.player_season_id = ps.player_season_id
WHERE ps.mins >= 900
GROUP BY pos.role_code
ORDER BY avg_assists_per90 DESC;

 * sqlite:////kaggle/input/datasets/tomersolo/sql-practice-football-players/sqlpractice.sqlite
Done.


role_code,n,avg_assists_per90
M,34469,0.153
AM,40784,0.14
DM,15107,0.138
ST,15045,0.103
WB,9341,0.088
D,38522,0.065
GK,5441,0.0
